# Road Accidents France — Exploratory Data Analysis

This notebook explores the cleaned BAAC dataset to answer key questions about road accident patterns in France.

**Questions explored:**
1. Peak hours for accidents
2. Severity by hour of day — are nighttime accidents more severe?
3. Seasonality — are there more accidents in summer or winter?
4. Severity by weather conditions
5. Severity by road type
6. Severity by vehicle type
7. Severity by age group
8. Severity by trip purpose
9. Geographic distribution of fatal accidents
10. Correlation matrix

**Data:** output of `cleaning.ipynb`

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

df = pd.read_csv('accidents_france_2023_clean_for_EDA.csv')
dftest = pd.read_csv('accidents_france_2024_clean_for_EDA.csv')
df_ml = pd.read_csv('accidents_france_2023_clean_for_ML.csv')

df['datetime'] = pd.to_datetime(df['datetime'])
dftest['datetime'] = pd.to_datetime(dftest['datetime'])

print(f"2023 dataset: {df.shape}")
print(f"2024 dataset: {dftest.shape}")

## 1. Peak Hours for Accidents

**Question:** Are there specific hours with higher accident frequency?

In [ ]:
df_by_hour_2023 = df.groupby('hour').size().reset_index(name='nb_accidents')
df_by_hour_2023['year'] = '2023'

dftest['hour'] = dftest['datetime'].dt.hour
df_by_hour_2024 = dftest.groupby('hour').size().reset_index(name='nb_accidents')
df_by_hour_2024['year'] = '2024'

px.line(pd.concat([df_by_hour_2023, df_by_hour_2024]),
        x='hour', y='nb_accidents', color='year',
        title='Number of accidents by hour — 2023 vs 2024', height=500)

**Observation:** Two clear peaks at 8am and 6pm, consistent across both years — corresponding to morning and evening commuting hours.

**Selection bias:** These peaks do not mean driving at these hours is *more dangerous* — there is simply more traffic. To assess actual risk, we would need to normalize by traffic volume at each hour.

**Consistency across years:** The near-identical patterns in 2023 and 2024 suggest this is a stable structural phenomenon, not a one-year anomaly.

## 2. Severity by Hour of Day

**Question:** Are nighttime accidents proportionally more severe?

In [ ]:
df_grav_hour = df.groupby('hour')['Injury_Severity'].value_counts(normalize=True).reset_index()

px.bar(df_grav_hour, x='hour', y='proportion', color='Injury_Severity',
       title='Injury severity proportion by hour of day', height=500)

**Observation:** The proportion of uninjured users is slightly lower at night (0h-5h), suggesting nighttime accidents are proportionally more severe. The share of fatalities appears slightly higher between midnight and 5am.

**Possible explanations:** Higher speeds at night, driver fatigue, reduced visibility, delayed emergency response.

**Limitation:** Differences are small. A chi-squared test would be needed to confirm statistical significance.

## 3. Seasonality — Monthly Accident Count

**Question:** Are there more accidents in summer or winter?

In [ ]:
df_by_month_2023 = df.groupby('month').size().reset_index(name='nb_accidents')
df_by_month_2023['year'] = '2023'

dftest['month'] = dftest['datetime'].dt.month
df_by_month_2024 = dftest.groupby('month').size().reset_index(name='nb_accidents')
df_by_month_2024['year'] = '2024'

month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

df_months = pd.concat([df_by_month_2023, df_by_month_2024])
df_months['month_name'] = df_months['month'].map(month_names)

px.line(df_months, x='month_name', y='nb_accidents', color='year',
        title='Number of accidents by month — 2023 vs 2024',
        category_orders={'month_name': list(month_names.values())},
        height=500)

## 4. Severity by Weather Conditions

**Question:** Does bad weather increase accident severity?

In [ ]:
df_grav_weather = df.groupby('Weather')['Injury_Severity'].value_counts(normalize=True).reset_index()

px.bar(df_grav_weather, x='Weather', y='proportion', color='Injury_Severity',
       title='Injury severity by weather conditions', height=500)

**Note on causality:** Even if certain weather conditions show higher fatality rates, we cannot conclude they *cause* more severe accidents. Confounding variables (speed, road type, driver behavior) may explain the observed differences.

## 5. Severity by Road Type

**Question:** Are motorways safer than departmental roads?

In [ ]:
df_grav_road = df.groupby('Road_Category')['Injury_Severity'].value_counts(normalize=True).reset_index()

px.bar(df_grav_road, x='Road_Category', y='proportion', color='Injury_Severity',
       title='Injury severity by road type', height=500)

## 6. Severity by Vehicle Type

**Question:** Do motorcyclists face higher fatality rates than car drivers?

In [ ]:
df_grav_veh = df.groupby('Vehicle_Category')['Injury_Severity'].value_counts(normalize=True).reset_index()

px.bar(df_grav_veh, x='Vehicle_Category', y='proportion', color='Injury_Severity',
       title='Injury severity by vehicle type', height=500)

## 7. Severity by Age Group

**Question:** Are elderly and young users more vulnerable to severe injuries?

In [ ]:
df['age_group'] = pd.cut(df['Age'],
                         bins=[0, 18, 25, 35, 45, 55, 65, 100],
                         labels=['<18', '18-25', '25-35', '35-45', '45-55', '55-65', '65+'])

df_grav_age = df.groupby('age_group')['Injury_Severity'].value_counts(normalize=True).reset_index()

px.bar(df_grav_age, x='age_group', y='proportion', color='Injury_Severity',
       title='Injury severity by age group', height=500)

## 8. Severity by Trip Purpose

**Question:** Are professional trips more dangerous than leisure trips?

In [ ]:
df_grav_trip = df.groupby('Trip_Purpose')['Injury_Severity'].value_counts(normalize=True).reset_index()

px.bar(df_grav_trip, x='Trip_Purpose', y='proportion', color='Injury_Severity',
       title='Injury severity by trip purpose', height=500)

## 9. Geographic Distribution of Accidents

**Question:** Are fatal accidents concentrated in specific regions?

In [ ]:
px.scatter(df, x='Longitude', y='Latitude', color='Injury_Severity',
           opacity=0.3, height=600,
           title='Geographic distribution of accidents by severity (2023)')

## 10. Correlation Matrix

**Question:** Are there linear relationships between numeric variables?

In [ ]:
df_num = df_ml.select_dtypes(include='number')
corr = df_num.corr()

px.imshow(corr, title='Correlation matrix (numeric features)', height=700)

**Observations:**
- `place` and `catu` are strongly correlated — seat position depends on user category
- `vma` and `agg` are negatively correlated — speed limits are lower in urban areas
- `vma` and `catr` are correlated — road type determines speed limits
- All correlations with `grav` (injury severity) are weak — no single variable alone explains severity

**Limitation:** Pearson correlation measures linear relationships only. Since most variables are categorical encoded as integers, a chi-squared test would be more appropriate.

The weak correlations with `grav` are consistent with the ML results in `ml.ipynb` — see conclusions there.